In [43]:
import requests
import populartimes
import pandas as pd

In [21]:
key = 'sua api key'

In [41]:
def get_popularidade(query, api_key):
    # 1. Buscar IDs de lugares com a query (scrap da API google places)
    url = 'https://places.googleapis.com/v1/places:searchText'
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': 'places.id'
    }
    data = {
        'textQuery': query
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code != 200: #check de resposta a API
        print(f"Erro ao buscar IDs: {response.status_code}, {response.text}")
        return []

    place_ids = [place['id'] for place in response.json().get('places', [])] #salvando todos resultados de IDs em uma lista

    # 2. Buscar dados de popularidade de cada ID
    resultados = []
    for pid in place_ids:
        try:
            dados = populartimes.get_id(api_key, pid)
            resultados.append(dados)
        except Exception as e:
            print(f"Erro com {pid}: {e}")
    
    return resultados

In [ ]:
dados = get_popularidade('academias Mococa', key)

In [59]:
registros = []

for lugar in dados:
    nome_local = lugar.get("name", "Desconhecido")
    endereco = lugar.get("address", "N/A")
    popularidade = lugar.get("populartimes", [])
    
    for dia in popularidade:
        dia_semana = dia['name']
        for hora, valor in enumerate(dia['data']):
            registros.append({ 
                "nome_local": nome_local,
                "endereco": endereco,
                "dia": dia_semana,
                "hora": hora,
                "popularidade": valor
            })

df = pd.DataFrame(registros)     
df.to_parquet('caminho para salvar')